In [ ]:
import pandas as pd
import os


# Load the Excel file
excel_file_path = '/Users/vi3680mu/Library/CloudStorage/GoogleDrive-murarosilva@gmail.com/My Drive/[1] Lund University/[1] Research Projects/Digital Public Infrastructures/Data/Overton/results-2025-05-19.xlsx'
excel_data = pd.read_excel(excel_file_path,sheet_name='Results')

In [ ]:
# Display the first few rows of each file
excel_data

In [ ]:
# Assuming the title columns are named 'Title' in both datasets
csv_titles = csv_data['title']
excel_titles = excel_data['Title']

# Find duplicates
duplicates = csv_titles[csv_titles.isin(excel_titles)]

# Display duplicates
duplicates

In [ ]:
excel_data.columns

In [ ]:
new_excel_data = excel_data[550:560].reset_index(drop=True)
new_excel_data

In [ ]:
# Check for commas in the 'PDF URL' column
comma_in_pdf_url = excel_data[excel_data['PDF URL'].str.contains(',', na=False)]

# Display rows with commas in 'PDF URL'
comma_in_pdf_url

In [ ]:
excel_data[559:562]['PDF URL'].to_list()

In [ ]:
import os
import requests
import urllib3
import pandas as pd
from requests.utils import requote_uri

# Suppress only the specific InsecureRequestWarning
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Define the root folder where the PDFs will be saved
root_folder = '/Users/vi3680mu/Library/CloudStorage/GoogleDrive-murarosilva@gmail.com/My Drive/[1] Lund University/[1] Research Projects/Digital Public Infrastructures/Data/PDFs_May_2025'
os.makedirs(root_folder, exist_ok=True)

# Add new columns to store results
excel_data['Download Status'] = ''
excel_data['Downloaded Filename'] = ''

# Get total number of rows for progress
total = len(excel_data)

# Start download loop
for index, row in excel_data.iterrows():
    print(f'[{index + 1}/{total}] Processing row...')

    # Filter only English documents
    if row['Languages'] != 'eng':
        excel_data.at[index, 'Download Status'] = 'Skipped (not English)'
        print(f'    ⏭ Skipped (Language = {row["Languages"]})')
        continue

    pdf_url = row['PDF URL']
    if pd.notna(pdf_url):
        try:
            # Encode the URL to avoid UnicodeDecodeError
            encoded_url = requote_uri(str(pdf_url))
            print(f'    🧪 Trying URL: {repr(encoded_url)}')

            try:
                response = requests.get(encoded_url, allow_redirects=True, timeout=10, verify=False)
            except UnicodeDecodeError as e:
                print(f'    ❌ Skipped due to UnicodeDecodeError: {e}')
                excel_data.at[index, 'Download Status'] = f'Skipped: UnicodeDecodeError'
                continue
            content_type = response.headers.get('Content-Type', '')

            if response.status_code == 200 and 'application/pdf' in content_type:
                pdf_filename = os.path.join(root_folder, f'document_{index}.pdf')
                with open(pdf_filename, 'wb') as pdf_file:
                    pdf_file.write(response.content)
                print(f'    ✔ Downloaded to {pdf_filename}')
                excel_data.at[index, 'Download Status'] = 'Success'
                excel_data.at[index, 'Downloaded Filename'] = f'document_{index}.pdf'
            else:
                print(f'    ✖ Failed (status: {response.status_code}, content-type: {content_type})')
                excel_data.at[index, 'Download Status'] = f'Failed (status {response.status_code})'

        except requests.exceptions.TooManyRedirects:
            print(f'    ⚠ Too many redirects - {pdf_url}')
            excel_data.at[index, 'Download Status'] = 'Too many redirects'
        except requests.exceptions.SSLError as e:
            print(f'    ⚠ SSL error - {e}')
            excel_data.at[index, 'Download Status'] = f'SSL error: {e}'
        except requests.exceptions.ReadTimeout:
            print(f'    ⌛ Read timeout')
            excel_data.at[index, 'Download Status'] = 'Read timeout'
        except requests.exceptions.RequestException as e:
            print(f'    ❌ Request failed - {e}')
            excel_data.at[index, 'Download Status'] = f'Request failed: {e}'
    else:
        print(f'    ⚠ No URL provided')
        excel_data.at[index, 'Download Status'] = 'No URL'

In [ ]:
# Save final download report
output_path = os.path.join(root_folder, 'download_report.xlsx')
excel_data.to_excel(output_path, index=False)
print(f'\n✅ Download report saved to: {output_path}')